# Titanic 生存预测项目 —— 完整流程模板

**目标**：走完 数据清洗 → 特征工程 → 建模 → 模型评估 → 可视化 这一整套流程，
每一步都留了 `# TODO` 和思考问题，帮你在做的时候顺便想清楚"为什么这么做"，
这样面试时才能讲清楚细节，而不只是"跑了个模型"。

**数据集**：Kaggle "Titanic - Machine Learning from Disaster"
下载地址：https://www.kaggle.com/competitions/titanic/data
下载 `train.csv`，放在这个 notebook 同一个文件夹下即可。

**建议耗时**：1-2天做完整个流程，愿意打磨可以花到3天（重点打磨"模型评估"部分，那是这个岗位JD最看重的）


## 1. 环境准备与数据加载

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, classification_report
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

df = pd.read_csv("train.csv")
df.head()


## 2. 数据探索（EDA）

先看整体情况：有多少行、哪些列有缺失值、目标变量（Survived）的分布是否均衡。

**思考题**：如果 Survived 里 0 和 1 的比例严重不均衡（比如 80/20），会对后面模型评估有什么影响？
（提示：这会决定你后面要不要特别关注 precision/recall，而不能只看 accuracy）


In [ ]:
df.info()
print("\n缺失值统计：")
print(df.isnull().sum())
print("\n目标变量分布：")
print(df["Survived"].value_counts(normalize=True))


In [ ]:
# 简单可视化：不同性别、舱位等级下的生存率
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df, x="Sex", y="Survived", ax=axes[0])
axes[0].set_title("Survival Rate by Sex")
sns.barplot(data=df, x="Pclass", y="Survived", ax=axes[1])
axes[1].set_title("Survival Rate by Pclass")
plt.tight_layout()
plt.show()


## 3. 数据清洗（缺失值处理）

`Age` 列有不少缺失值，`Cabin` 缺失更严重，`Embarked` 缺失很少。

**这是面试最容易被问细节的地方**，处理方式没有唯一正确答案，但你要能说清楚"为什么这么选"：
- 用整体中位数填充 Age？简单但忽略了不同 Pclass/Sex 群体年龄本来就不同的事实
- 按 Pclass + Sex 分组后取组内中位数填充？更精细，但要说清楚为什么这个分组有道理
- Cabin 缺失太多，直接删列，还是提取"是否有Cabin记录"作为一个新特征？

下面代码用了"按分组填充"的方式，你可以试试改成"整体中位数"对比一下最终模型效果有没有差别——
这个对比本身就是一个很好的"我做了什么权衡"的故事。


In [ ]:
# 按 Pclass + Sex 分组，用组内中位数填充 Age
df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(lambda x: x.fillna(x.median()))

# Embarked 缺失很少，直接用众数填充
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Cabin 缺失太多，转成"是否有Cabin记录"这个二元特征，而不是直接删掉信息
df["HasCabin"] = df["Cabin"].notnull().astype(int)

# 确认缺失值处理完毕
df[["Age", "Embarked", "Cabin", "HasCabin"]].isnull().sum()


## 4. 特征工程

从 `Name` 里提取称谓（Mr/Mrs/Miss/Master等），这个称谓其实隐含了年龄段和社会地位信息，
往往比原始 Name 本身更有预测力——这是一个很好的"我不是照搬模板，而是理解数据"的例子。

同时把 `SibSp`（兄弟姐妹/配偶数）和 `Parch`（父母/子女数）合并成 `FamilySize`，
思考："一个人的家庭规模大小，可能怎样影响他/她逃生的优先级或行为？"


In [ ]:
# 提取称谓
df["Title"] = df["Name"].str.extract(r",\s*([^\.]*)\.")
# 把稀有称谓合并成 "Rare"，避免类别过于稀疏
title_counts = df["Title"].value_counts()
rare_titles = title_counts[title_counts < 10].index
df["Title"] = df["Title"].replace(rare_titles, "Rare")
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})

# 家庭规模
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df[["Name", "Title", "SibSp", "Parch", "FamilySize", "IsAlone"]].head()


In [ ]:
# 类别变量编码
df_model = pd.get_dummies(
    df[["Survived", "Pclass", "Sex", "Age", "Fare", "Embarked",
        "Title", "FamilySize", "IsAlone", "HasCabin"]],
    columns=["Pclass", "Sex", "Embarked", "Title"],
    drop_first=True
)
df_model.head()


## 5. 划分训练/测试集 + 建模（先跑 baseline，再对比）

先用逻辑回归做一个简单的 baseline，再用随机森林对比。
**为什么要有 baseline**：如果面试官问"你怎么知道随机森林比逻辑回归好"，
你需要有一个基准模型的数字来做对比，而不是凭感觉说"随机森林更好"。


In [ ]:
X = df_model.drop(columns=["Survived"])
y = df_model["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline: 逻辑回归
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
y_pred_log = log_reg.predict(X_test_scaled)

# 对比模型: 随机森林
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)  # 树模型不需要标准化
y_pred_rf = rf.predict(X_test)


## 6. 模型评估（这是全流程里最重要的一步，对应岗位JD"评估模型表现"）

**关键点：不要只看 accuracy。**

想一想：在 Titanic 这个场景里，
- "预测会生还，但实际遇难"（False Positive）
- "预测会遇难，但实际生还"（False Negative）

哪个错误的代价更值得关注？这个问题在真实业务场景里（比如信用风险、疾病诊断）会更明显，
现在用 Titanic 练习这种思考方式。


In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    print(f"===== {model_name} =====")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred):.3f}")
    print()
    print(classification_report(y_true, y_pred))

evaluate_model(y_test, y_pred_log, "Logistic Regression")
evaluate_model(y_test, y_pred_rf, "Random Forest")


In [ ]:
# 混淆矩阵可视化对比
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_log, ax=axes[0], cmap="Blues")
axes[0].set_title("Logistic Regression")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=axes[1], cmap="Blues")
axes[1].set_title("Random Forest")
plt.tight_layout()
plt.show()


In [ ]:
# ROC 曲线对比
y_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]

fpr_log, tpr_log, _ = roc_curve(y_test, y_prob_log)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.plot(fpr_log, tpr_log, label=f"Logistic Regression (AUC={roc_auc_score(y_test, y_prob_log):.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={roc_auc_score(y_test, y_prob_rf):.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


## 7. 特征重要性 + 业务洞察可视化

这一步对应 JD 里"识别并可视化关键洞察"这条要求：
不只是给出一个数字，而是把结果"翻译"成一句人话结论。


In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values[:10], y=importances.index[:10])
plt.title("Top 10 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.show()

print(importances.head(10))


## 8. 写结论（简历/面试用得上的总结）

**TODO：根据你实际跑出来的结果，填写下面几句话**（这几句话稍加整理，
就是你简历project描述的原材料）：

1. 数据集规模、缺失值情况、你选择的处理方式及原因：___
2. baseline（逻辑回归）vs 对比模型（随机森林）的关键指标对比：___
3. 哪个特征最重要，这个结果是否符合直觉，说明了什么：___
4. 如果要改进，你会尝试什么（比如调参、加交叉验证、尝试XGBoost、处理类别不平衡）：___

---

### 下一步可以做的加分项（如果时间允许）
- 加上 **K折交叉验证**（`cross_val_score`），让结果更稳健，也是一个常见面试考点
- 试试处理**样本不平衡**（虽然Titanic不算严重不平衡，但可以练手 `class_weight="balanced"` 或 SMOTE）
- 用 `GridSearchCV` 做简单的超参数搜索，讲清楚你搜了哪些参数、为什么
